# पाठ 12 - एजेंट स्क्रैचपैड के साथ चैट इतिहास में कमी

यह नोटबुक माइक्रोसॉफ्ट एजेंट फ्रेमवर्क का उपयोग करके लंबे संवादों में संदर्भ कैसे प्रबंधित करें, यह दिखाती है। जैसे-जैसे बातचीत बढ़ती है, टोकन की संख्या बढ़ती है — अंततः मॉडल की संदर्भ विंडो से अधिक हो जाती है। हम इस समस्या को **संदर्भ सारांशण पैटर्न** और **एजेंट स्क्रैचपैड** द्वारा संबोधित करते हैं जो स्थायी मेमोरी प्रदान करता है।

## आप क्या सीखेंगे:
1. **संदर्भ प्रबंधन क्यों महत्वपूर्ण है**: टोकन सीमाएँ और संदर्भ विंडोज़ की समझ
2. **संदर्भ-सचेत एजेंट**: ऐसे एजेंट बनाना जो अपनी स्वयं की बातचीत संदर्भ को प्रबंधित करें
3. **संदर्भ सारांशण पैटर्न**: बातचीत के इतिहास को संक्षिप्त करने के लिए टूल्स का उपयोग
4. **एजेंट स्क्रैचपैड**: स्थायी मेमोरी जो संदर्भ कमी को सहन करती है

## आवश्यकताएँ:
- पर्यावरण चर कॉन्फ़िगर के साथ Azure OpenAI सेटअप
- पिछले पाठों से मूल एजेंट अवधारणाओं की समझ


## सेटअप


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv --quiet

In [ ]:
import os
import asyncio
import dotenv
from datetime import datetime
from pathlib import Path

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

In [ ]:
dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

print("Microsoft Foundry client configured")

## संदर्भ प्रबंधन क्यों महत्वपूर्ण है

हर LLM का एक सीमित **संदर्भ विंडो** होता है — एक एकल अनुरोध में वह अधिकतम टोकन की संख्या है जिसे वह संसाधित कर सकता है। जैसे-जैसे बहु-चरणीय बातचीत आगे बढ़ती है:

- **टोकन गणना रैखिक रूप से बढ़ती है** प्रत्येक उपयोगकर्ता संदेश और सहायक प्रतिक्रिया के साथ।
- **प्रॉम्प्ट टोकन लागत पर हावी होते हैं** क्योंकि पूरा इतिहास हर चरण में पुनः भेजा जाता है।
- अंततः बातचीत **संदर्भ विंडो से बाहर निकल जाती है** और मॉडल या तो ट्रंकेट कर देता है या त्रुटि देता है।

### संदर्भ प्रबंधन के लिए रणनीतियाँ

| रणनीति | यह कैसे काम करता है | ट्रेड-ऑफ |
|---|---|---|
| **ट्रंकेशन** | सबसे पुराने संदेश हटाएं | प्रारंभिक संदर्भ खो जाता है |
| **सारांश** | पुराने संदेशों को सारांश में संक्षेपित करें | कुछ विवरण खो जाता है, लेकिन मुख्य बिंदु बने रहते हैं |
| **स्क्रैचपैड / बाहरी मेमोरी** | बातचीत के बाहर प्रमुख तथ्य संग्रहीत करें | टूल कॉल की आवश्यकता होती है, लेकिन किसी भी कमी से बचता है |

इस नोटबुक में हम **सारांश** को एक **स्क्रैचपैड टूल** के साथ मिलाते हैं ताकि एजेंट तब भी निरंतरता बनाए रख सके जब बातचीत का इतिहास संक्षिप्त हो जाए।


## एक संदर्भ-सचेत एजेंट बनाना


In [ ]:
agent = client.as_agent(
    name="ContextAwareAgent",
    instructions="""You are a helpful travel planning assistant with excellent memory management.
When conversations get long:
1. Summarize previous context into key points
2. Track user preferences mentioned earlier
3. Reference previous decisions without repeating full details
Always maintain continuity while being concise.""",
)

print("Context-aware travel planning agent created")

## एक लंबी बातचीत का सिमुलेशन

चलिए एक मल्टी-टर्न बातचीत के माध्यम से चलते हैं ताकि देखें कि संदर्भ कैसे जमा होता है। एजेंट को टर्न्स के दौरान प्रमुख विवरण (पसंद, बजट, यात्रा की तारीखें) को बनाए रखना चाहिए और निरंतरता का प्रदर्शन करना चाहिए।


In [ ]:
session = agent.create_session()

# Turn 1 - Initial preferences
response = await agent.run("I'm planning a trip to Japan. I love sushi, temples, and photography.", session=session)
print(f"Turn 1: {response}\n")

# Turn 2 - More details
response = await agent.run("My budget is $3000 and I'll be traveling solo for 10 days in April.", session=session)
print(f"Turn 2: {response}\n")

# Turn 3 - Test context retention
response = await agent.run("Based on everything I've told you so far, what's the one thing you'd recommend I not miss?", session=session)
print(f"Turn 3: {response}\n")

ध्यान दें कि एजेंट पहले के दौरों से संदर्भ को कैसे बनाए रखता है — यह जापान, सुशी, मंदिरों, फोटोग्राफी, $3000 बजट, अकेले यात्रा, और अप्रैल के समय के बारे में जानता है। एक संक्षिप्त बातचीत में यह अच्छा काम करता है, लेकिन जैसे-जैसे बातचीत बढ़ती है, पूरी इतिहास को पुनः भेजना महंगा हो जाता है।

चलिए अधिक दौरों के साथ बातचीत जारी रखते हैं ताकि संदर्भ संचय को देखा जा सके:


In [ ]:
# Turn 4 - Expand the conversation
response = await agent.run("What about accommodation? I prefer traditional Japanese inns.", session=session)
print(f"Turn 4: {response}\n")

# Turn 5 - Change of plans
response = await agent.run("Actually, I've changed my mind about the dates. I'll go in October instead for the autumn colors.", session=session)
print(f"Turn 5: {response}\n")

# Turn 6 - Test retention after change
response = await agent.run("Summarize my complete travel plan so far — destination, budget, duration, interests, accommodation, and timing.", session=session)
print(f"Turn 6: {response}\n")

## संदर्भ सारांशण पैटर्न

जैसे-जैसे बातचीत बढ़ती है, हम संचित संदर्भ को एक संक्षिप्त स्वरूप में संक्षेपित करने के लिए एक **सारांशण उपकरण** का उपयोग कर सकते हैं। एजेंट इस उपकरण को प्रमुख प्राथमिकताओं को रिकॉर्ड करने के लिए कॉल करता है ताकि पुरानी संदेशों को हटाए जाने पर भी आवश्यक जानकारी सुरक्षित रहे।

यह पैटर्न अधिक परिष्कृत इतिहास संक्षेपण के लिए आधार स्तंभ है:
1. एजेंट बातचीत से प्रमुख तथ्यों की पहचान करता है
2. यह उन्हें स्थायी बनाने के लिए सारांशण उपकरण को कॉल करता है
3. पुराने संदेश सुरक्षित रूप से हटाए जा सकते हैं क्योंकि सारांश आवश्यक बातों को कैप्चर करता है

नीचे हमने `summarize_preferences` नामक एक उपकरण परिभाषित किया है जिसे एजेंट सीखी गई जानकारियों का एक संक्षिप्त सार रिकॉर्ड करने के लिए कॉल कर सकता है।


In [ ]:
@tool(approval_mode="never_require")
def summarize_preferences(conversation_notes: str) -> str:
    """Summarize accumulated user preferences into a compact format."""
    return f"[SUMMARY] User preferences recorded: {conversation_notes}"


# Create an enhanced agent with the summarization tool
summarizing_agent = client.as_agent(
    name="SummarizingTravelAgent",
    instructions="""You are a helpful travel planning assistant that actively manages conversation context.

CONTEXT MANAGEMENT RULES:
1. After gathering several user preferences, call summarize_preferences() to record a compact summary
2. When the user asks you to recall details, reference your recorded summaries
3. Keep responses concise — avoid restating the entire history

PLANNING PROCESS:
1. Gather user preferences (destination, budget, dates, interests)
2. Summarize preferences using the tool
3. Create recommendations based on the summary
4. Update the summary when preferences change""",
    tools=[summarize_preferences],
)

print("Summarizing travel agent created with context tools")

In [ ]:
# Demonstrate the summarization pattern
summary_session = summarizing_agent.create_session()

# Provide a batch of preferences
response = await summarizing_agent.run(
    "I want to visit Greece. I love seafood, history, and island hopping. "
    "Budget is $4000 for two weeks. Traveling with my partner in June. "
    "Please record these preferences using your summarization tool.",
    session=summary_session,
)
print(f"Agent: {response}\n")

# Ask the agent to use the recorded context
response = await summarizing_agent.run(
    "Now, based on what you've recorded, suggest the top 3 islands we should visit.",
    session=summary_session,
)
print(f"Agent: {response}\n")

## सारांश

इस पाठ में आपने Microsoft Agent Framework का उपयोग करके लंबी अवधि चलने वाली एजेंट वार्तालापों में संदर्भ को कैसे प्रबंधित करें, यह सीखा:

### मुख्य अवधारणाएँ
- **संदर्भ विंडो सीमित हैं** — वार्तालाप इतिहास में प्रत्येक टोकन की कीमत होती है और यह सीमा की ओर जाता है।
- **सारांश उपकरण** एजेंट को संग्रहीत संदर्भ को संक्षिप्त सारांशों में समेटने देते हैं, जिससे टोकन उपयोग कम होता है और आवश्यक जानकारी संरक्षित रहती है।
- **एजेंट स्क्रैचपैड** एक स्थायी बाहरी मेमोरी प्रदान करता है जो किसी भी वार्तालाप संक्षेपण से सुरक्षित रहता है।

### आपने क्या बनाया
- एक **संदर्भ-सचेत एजेंट** जो बहु-टर्न बातचीत में निरंतरता बनाए रखता है
- एक **सारांश उपकरण** (`summarize_preferences`) जो उपयोगकर्ता के महत्वपूर्ण विवरणों को संक्षिप्त रूप में रिकॉर्ड करता है
- एक **बहु-टर्न बातचीत** जो संदर्भ को बनाए रखती है और परिवर्तन को संभालती है

### वास्तविक दुनिया के अनुप्रयोग
- **कस्टमर सर्विस बॉट्स**: लंबे सपोर्ट सत्रों में प्राथमिकताएँ याद रखें
- **पर्सनल असिस्टेंट्स**: संदर्भ को फिर से समझाए बिना चल रही परियोजनाओं को ट्रैक करें
- **शैक्षिक ट्यूटर**: कई इंटरैक्शन के दौरान छात्र की प्रगति बनाए रखें

### अगले कदम
- फ़ाइल-आधारित स्थायित्व के साथ एक पूर्ण स्क्रैचपैड टूल लागू करें
- सारांश बनाने के बाद स्वचालित इतिहास कटौती जोड़ें
- अर्थपूर्ण मेमोरी खोज के लिए वेक्टर डेटाबेस के साथ संयोजन करें
- ऐसे एजेंट बनाएं जो पूरी संदर्भ के साथ दिनों बाद वार्तालाप फिर से शुरू कर सकें


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**अस्वीकरण**:
इस दस्तावेज़ का अनुवाद AI अनुवाद सेवा [Co-op Translator](https://github.com/Azure/co-op-translator) का उपयोग करके किया गया है। जबकि हम सटीकता के लिए प्रयास करते हैं, कृपया ध्यान दें कि स्वचालित अनुवादों में त्रुटियाँ या अशुद्धियाँ हो सकती हैं। मूल दस्तावेज़ अपनी मूल भाषा में ही प्रामाणिक स्रोत माना जाना चाहिए। महत्वपूर्ण जानकारी के लिए, पेशेवर मानव अनुवाद की सिफारिश की जाती है। इस अनुवाद के उपयोग से उत्पन्न किसी भी गलतफहमी या गलत व्याख्या के लिए हम उत्तरदायी नहीं हैं।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
